# Optuna run metadata by model

Read this report by **model → framework → phase → concurrency → objective**. Model groups use the W&B display-name suffix (including `cpm_2` / `cmp_2`); DSpark groups remain separate. Unsuffixed runs fall back to model metadata. Additional variants are shown after the eight requested models.

The notebook reconciles local SQLite studies, all cached W&B exports, and optional live W&B data by **W&B ID**. Set `LOAD_WANDB=True` to refresh all project runs and full histories; the default loads the refreshed snapshot offline. Every source and excluded identity is auditable below.

Only measured runs with at least **300 seconds** of recorded optimization time enter objective reports. Terminal W&B runtime is a fallback and includes logging overhead. Unknown duration, synthetic data and unsupported frameworks are excluded. Startup coverage also lists shorter runs so that available initiation evidence remains visible.

Objective tables report **mean and sample variance (ddof=1)** per framework and for each model overall, separated by phase, concurrency, objective, metric and data coverage. One observation is one eligible completed trial; constrained studies require SLO-feasible objective results; W&B best/Pareto-only summaries are labeled separately and never pooled with full trial histories. Pareto studies report both paired objectives. Overall values pool observations across frameworks within the displayed strata; differing workloads/search budgets can affect them. Singleton variance is unavailable, not zero. Per-context plots retain workload/environment provenance.

Startup tables show recorded launch-to-ready (server initiation including model loading), client preparation, warmup, benchmark and runner wall time. Explicit engine-log weight loading, compilation and graph-capture observations stay separate: they may overlap and must not be summed into wall time. Partial phases stay in the detailed table but are excluded from timing means. Missing telemetry is unavailable, never estimated from run duration.

Optuna PED-ANOVA importance is calculated per run and objective from complete local or W&B trial histories, then displayed by model and framework. Remote search spaces are reconstructed from the declared sampling menu plus observed choices; no importance is inferred from best/Pareto summaries. `s22` is seed 22 and `r22` is repeat 22.


## Local data and analysis settings


In [ ]:
from pathlib import Path
from collections import Counter
from contextlib import closing
import json
import html
import math
import sqlite3
import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "experiment_results").exists():
    ROOT = ROOT.parent
assert (ROOT / "experiment_results").exists(), "Run from the repository root or docs directory"
RUN_ROOTS = [ROOT / "experiment_results", ROOT / "results"]
MIN_RUN_SECONDS = 300
INCLUDE_SYNTHETIC = False
# Cached W&B results are loaded automatically; enable this only to refresh live data.
LOAD_WANDB = False
WANDB_SNAPSHOT = ROOT / "experiment_results" / "wandb_run_snapshot.json"
WANDB_PROJECT = "maorblumberg-tel-aviv-university/inference-optimization"
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 160)
import sys
sys.path.insert(0, str(ROOT / "docs"))
from optuna_metadata_analysis import (
    MODEL_ORDER, model_group, load_export, objective_statistics,
    startup_rows, startup_statistics, startup_trial_table, table_html, history_importance, write_report_html,
)
WANDB_EXPORTS = ROOT / "results" / "sorted_results"


## Duration selection rules


In [ ]:
def seconds(value):
    try:
        value = float(value)
        return value if math.isfinite(value) and value >= 0 else None
    except (TypeError, ValueError):
        return None


def duration_for(attrs, trials, remote_summary=None, remote_state=None):
    for label, value in [
        ("local optimization_wall_seconds", attrs.get("optimization_wall_seconds")),
        ("W&B optimization_wall_seconds", (remote_summary or {}).get("study", {}).get("optimization_wall_seconds", (remote_summary or {}).get("optimization_wall_seconds"))),
    ]:
        duration = seconds(value)
        if duration is not None:
            return duration, label
    started = [pd.Timestamp(t["datetime_start"]) for t in trials if t["datetime_start"]]
    ended = [pd.Timestamp(t["datetime_complete"]) for t in trials if t["datetime_complete"]]
    terminal = trials and all(t["state"] in {"COMPLETE", "FAIL", "PRUNED"} for t in trials)
    if terminal and started and ended:
        return seconds((max(ended) - min(started)).total_seconds()), "local terminal trial span"
    if remote_state in {"finished", "failed", "crashed", "killed"}:
        duration = seconds((remote_summary or {}).get("_runtime"))
        if duration is not None:
            return duration, "W&B terminal _runtime"
    return None, "unknown"


records, issues = [], []
for database in sorted({p for root in RUN_ROOTS for p in root.rglob("*.sqlite3")} |
                       {p for root in RUN_ROOTS for p in root.rglob("*.db")}):
    try:
        with closing(sqlite3.connect(database.resolve().as_uri() + "?mode=ro", uri=True)) as db:
            tables = {r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
            if not {"studies", "study_user_attributes", "trials"} <= tables:
                continue
            for study_id, name in db.execute("SELECT study_id, study_name FROM studies"):
                attrs = {k: json.loads(v) for k, v in db.execute(
                    "SELECT key, value_json FROM study_user_attributes WHERE study_id=?", (study_id,))}
                contract = attrs.get("contract", {})
                if contract.get("phase") not in {"A", "B"}:
                    issues.append({"source": str(database), "issue": f"No Phase A/B contract: {name}"})
                    continue
                trials = [dict(zip(["state", "datetime_start", "datetime_complete"], row))
                          for row in db.execute("SELECT state, datetime_start, datetime_complete FROM trials WHERE study_id=?", (study_id,))]
                files = sorted(database.parent.glob(f"{name}-*-measured.jsonl"))
                measurements = valid = 0
                for file in files:
                    for line_number, line in enumerate(file.open(), 1):
                        if not line.strip():
                            continue
                        try:
                            result = json.loads(line).get("results", {})
                            measurements += 1
                            valid += result.get("benchmark_valid") is True
                        except (ValueError, TypeError):
                            issues.append({"source": str(file), "issue": f"Invalid JSONL line {line_number}"})
                duration, timing = duration_for(attrs, trials)
                locator = attrs.get("wandb", {})
                records.append(dict(study_name=name, phase=contract["phase"], framework=contract.get("framework"),
                    contract=contract, attrs=attrs, trials=trials, database=str(database.relative_to(ROOT)),
                    measured_files=[str(p.relative_to(ROOT)) for p in files], measurement_rows=measurements,
                    valid_measurement_rows=valid, duration_seconds=duration, duration_source=timing,
                    wandb_id=locator.get("id"), wandb_url=locator.get("url"), sources="local", remote_state=None))
    except (sqlite3.Error, ValueError) as exc:
        issues.append({"source": str(database), "issue": type(exc).__name__})
print(f"Discovered {len(records)} local Phase A/B studies")

## W&B sources
The original snapshot and all files under `results/sorted_results` are loaded automatically. Newer exports carry complete trial histories, including execution telemetry. IDs are deduplicated, with the richest available history retained. The live refresh queries every run in the configured project and scans its history; authentication uses the existing W&B login. Cached data remains usable if a refresh fails. Source names and export timestamps are visible in the inventory and loading output.


## Cached W&B loading and source reconciliation


In [ ]:
from types import SimpleNamespace


def normalize_wandb_summary(summary):
    # W&B can serialize nested study summaries as flattened dotted keys.
    result = dict(summary)
    study = dict(result.get("study") or {})
    for key, value in summary.items():
        if key.startswith("study."):
            parts = key.split(".")[1:]
            node = study
            for part in parts[:-1]:
                node = node.setdefault(part, {})
            node[parts[-1]] = value
    result["study"] = study
    return result


def read_wandb_snapshot(path, project):
    snapshot = json.loads(path.read_text())
    if snapshot.get("schema_version") != 1 or snapshot.get("project") != project:
        raise ValueError("Unexpected snapshot schema or W&B project")
    runs = []
    for row in snapshot["runs"]:
        run = dict(row)
        run["summary"] = normalize_wandb_summary(row["summary"])
        runs.append(SimpleNamespace(**run))
    return runs, snapshot["exported_at"]


exported_at = None
remote_by_id, remote_sources = {}, {}
# The original snapshot remains readable; newer exports add full trial histories.
if WANDB_SNAPSHOT.exists():
    try:
        cached_runs, exported_at = read_wandb_snapshot(WANDB_SNAPSHOT, WANDB_PROJECT)
        for run in cached_runs:
            remote_by_id[run.id] = run
            remote_sources[run.id] = "wandb snapshot"
        print(f"Loaded {len(cached_runs)} cached W&B runs from {WANDB_SNAPSHOT.relative_to(ROOT)} (exported {exported_at})")
    except (ValueError, KeyError, TypeError) as exc:
        issues.append({"source": str(WANDB_SNAPSHOT), "issue": f"Invalid W&B snapshot ({type(exc).__name__})"})
else:
    issues.append({"source": str(WANDB_SNAPSHOT), "issue": "W&B snapshot missing"})
for export in sorted(WANDB_EXPORTS.glob("*.json"), key=lambda p: p.stat().st_mtime):
    try:
        for row in load_export(export, WANDB_PROJECT):
            row["summary"] = normalize_wandb_summary(row["summary"])
            previous = remote_by_id.get(row["id"])
            if previous is not None and remote_sources.get(row["id"]) == "wandb snapshot" and hasattr(previous, "history"):
                continue  # A freshly fetched full snapshot takes precedence over older exports.
            if previous is not None and len(getattr(previous, "history", [])) > len(row["history"]):
                row["history"] = previous.history
            remote_by_id[row["id"]] = SimpleNamespace(**row)
            remote_sources[row["id"]] = str(export.relative_to(ROOT))
    except (ValueError, KeyError, TypeError) as exc:
        issues.append({"source": str(export), "issue": f"Invalid W&B export ({type(exc).__name__})"})
print(f"Cached unique W&B runs: {len(remote_by_id)}")
if LOAD_WANDB:
    try:
        import wandb
        live_runs = list(wandb.Api(timeout=30).runs(WANDB_PROJECT))
        for run in live_runs:
            history = []
            try:
                from optuna_metadata_analysis import normalize_trial
                for item in run.scan_history(page_size=1000):
                    trial = normalize_trial(item, run.config.get("metrics", []))
                    if trial["number"] is not None:
                        history.append(trial)
            except Exception as exc:
                issues.append({"source": run.url, "issue": f"W&B history unavailable ({type(exc).__name__})"})
                history = getattr(remote_by_id.get(run.id), "history", [])
            remote_by_id[run.id] = SimpleNamespace(id=run.id, name=run.name, url=run.url,
                state=run.state, group=run.group, config=dict(run.config),
                summary=normalize_wandb_summary(dict(run.summary)), history=history)
            remote_sources[run.id] = "wandb live"
        from datetime import datetime, timezone
        refreshed = {"schema_version": 1, "project": WANDB_PROJECT,
            "exported_at": datetime.now(timezone.utc).isoformat(),
            "runs": [vars(remote_by_id[run.id]) for run in live_runs]}
        WANDB_SNAPSHOT.write_text(json.dumps(refreshed, indent=2, default=str) + "\n")
        exported_at = refreshed["exported_at"]
        print(f"Read and cached {len(live_runs)} live W&B runs; live data replaces matching cached IDs")
    except Exception as exc:
        issues.append({"source": WANDB_PROJECT, "issue": f"Live W&B unavailable ({type(exc).__name__}); cached runs retained"})
        print("Live W&B unavailable; using the local snapshot and SQLite studies.")
else:
    print("Live W&B refresh disabled; cached results require no login or network connection.")
remote_runs = list(remote_by_id.values())

local_by_id = {}
for record in records:
    if record["wandb_id"]:
        local_by_id.setdefault(record["wandb_id"], []).append(record)
for run in remote_runs:
    contract, summary = dict(run.config), normalize_wandb_summary(dict(run.summary))
    history = getattr(run, "history", [])
    matches = local_by_id.get(run.id, [])
    if matches:
        for record in matches:
            record.update(display_name=run.name, remote_history=history, sources="local + " + remote_sources[run.id], wandb_url=run.url, remote_state=run.state, remote_summary=summary)
            record["duration_seconds"], record["duration_source"] = duration_for(record["attrs"], record["trials"], summary, run.state)
            for key in ["phase", "framework", "target_model", "concurrency", "workload", "sampler"]:
                if key in contract and contract[key] != record["contract"].get(key):
                    issues.append({"source": run.url, "issue": f"Local/W&B contract mismatch: {key}"})
    else:
        duration, timing = duration_for({}, [], summary, run.state)
        records.append(dict(study_name=contract.get("study_name", run.name), phase=contract.get("phase", "unknown"),
            framework=contract.get("framework"), contract=contract, attrs={}, trials=[], database=None,
            measured_files=[], measurement_rows=None, valid_measurement_rows=None,
            duration_seconds=duration, duration_source=timing, wandb_id=run.id, wandb_url=run.url,
            sources=remote_sources[run.id], remote_state=run.state, remote_summary=summary,
            display_name=run.name, remote_history=history))

# Duplicate copies of one W&B run contribute once, keeping the richest local copy.
unique = {}
for record in sorted(records, key=lambda r: len(r["trials"]), reverse=True):
    identity = record["wandb_url"] or (record["database"], record["study_name"])
    if identity in unique:
        issues.append({"source": record["database"], "issue": f"Duplicate run copy excluded: {record['study_name']}"})
    else:
        unique[identity] = record
records = list(unique.values())
for record in records:
    record["model_group"] = model_group(record.get("display_name", record["study_name"]), record["contract"])
records.sort(key=lambda r: (MODEL_ORDER.index(r["model_group"]) if r["model_group"] in MODEL_ORDER else len(MODEL_ORDER),
                            r["framework"] or "", r.get("display_name", r["study_name"]), r["wandb_id"] or ""))

## Five-minute run selection


In [ ]:
def exclusion(record):
    if record["framework"] not in {"vllm", "sglang", "trtllm"}:
        return "missing or unsupported framework"
    if record["contract"].get("environment", {}).get("measurement") == "SYNTHETIC" and not INCLUDE_SYNTHETIC:
        return "synthetic"
    if record["duration_seconds"] is None:
        return "unknown duration"
    if record["duration_seconds"] < MIN_RUN_SECONDS:
        return "under five minutes"
    return None

included = [r for r in records if exclusion(r) is None]
excluded = [r for r in records if exclusion(r) is not None]
RUN_COLUMNS = ["model_group", "display_name", "phase", "framework", "study_name", "duration_minutes", "duration_source", "sources", "database", "wandb_id", "wandb_url", "remote_state", "run_group", "measured_files"]

def manifest(rows):
    return pd.DataFrame([{**{k: r.get(k) for k in RUN_COLUMNS},
                          "duration_minutes": None if r["duration_seconds"] is None else r["duration_seconds"] / 60}
                         for r in rows], columns=RUN_COLUMNS)

# Preserve original experiment group; identical study names can have distinct W&B IDs.
remote_groups = {run.id: run.group for run in remote_runs}
for record in records:
    record["run_group"] = remote_groups.get(record["wandb_id"]) or (
        Path(record["database"]).parent.name if record["database"] else "unknown")

# This cell is the complete list of runs used by every summary below.
runs_used = manifest(included)
print(f"Runs used: {len(included)}; cutoff: {MIN_RUN_SECONDS / 60:g} minutes")

## Recorded run metadata


In [ ]:
def metadata_row(record):
    c = record["contract"]
    env = c.get("environment", {})
    states = dict(Counter(t["state"] for t in record["trials"])) if record["database"] else record.get("remote_summary", {}).get("study", {}).get("states", {})
    return dict(model_group=record["model_group"], phase=record["phase"], framework=record["framework"], study_name=record["study_name"],
        model=c.get("target_model", env.get("model")), concurrency=c.get("concurrency"),
        workload=c.get("workload"), measurement=env.get("measurement"), gpu=env.get("gpu_type"),
        gpu_count=env.get("gpu_count"), package_versions=env.get("package_versions"),
        framework_version=env.get("framework_version"), cuda_version=env.get("cuda_version"),
        driver_version=env.get("driver_version"), model_revision=env.get("model_revision"),
        sampler=c.get("sampler"), sampler_settings=c.get("sampler_settings"),
        target=c.get("target"), metrics=c.get("metrics"), directions=c.get("directions"),
        space_kind=c.get("space_kind"), sampling_menu=c.get("sampling_menu"),
        pruner=c.get("pruner_class"), budget_trials=c.get("budget_trials"),
        budget_gpu_seconds=c.get("budget_gpu_seconds"), constrained=c.get("constrained"),
        ttft_limit_ms=c.get("ttft_limit_ms"), e2e_limit_ms=c.get("e2e_limit_ms"),
        quick_demo=c.get("quick_demo"), trials=len(record["trials"]) if record["database"] else None,
        trial_states=states, measurement_rows=record["measurement_rows"],
        valid_measurement_rows=record["valid_measurement_rows"], duration_minutes=record["duration_seconds"] / 60)

metadata = pd.DataFrame([metadata_row(r) for r in included])

def distinct(values):
    return sorted({json.dumps(v, sort_keys=True, default=str) for v in values})

## Actual study results by objective

Objective sections show concise run tables with sampler, sampled trial parameters, optimization runtime, and best objective value. Pareto tables retain the paired objective values for each attainable candidate. Objective history plots use local trial records. Retrospective Optuna parameter importance uses the Ped-ANOVA evaluator with objective directions normalized from each stored Optuna study, where history and parameter variation are sufficient; W&B summaries alone do not contain full history.


## Objective trial results and study identities


In [ ]:
def decode_param(value, distribution_json):
    distribution = json.loads(distribution_json)
    if distribution["name"] == "CategoricalDistribution":
        return distribution["attributes"]["choices"][int(value)]
    return int(value) if distribution["name"] == "IntDistribution" else value


RESULT_STUDY = "B-vllm-c128-pareto_throughput-gp-s11-r3"
result_rows, result_issues = [], []
for record in included:
    contract = record["contract"]
    metrics = contract.get("metrics", [])
    directions = [str(d).upper() for d in contract.get("directions", [])]
    if not metrics or len(metrics) != len(directions) or any(d not in {"MINIMIZE", "MAXIMIZE"} for d in directions):
        result_issues.append({"study_name": record["study_name"], "issue": "Missing or unsupported objective contract"})
        continue
    candidates = []
    if record["database"]:
        with closing(sqlite3.connect((ROOT / record["database"]).resolve().as_uri() + "?mode=ro", uri=True)) as db:
            study_id = db.execute("SELECT study_id FROM studies WHERE study_name=?", (record["study_name"],)).fetchone()[0]
            for trial_id, number, end in db.execute("SELECT trial_id, number, datetime_complete FROM trials WHERE study_id=? AND state='COMPLETE' ORDER BY number", (study_id,)):
                attrs = {k: json.loads(v) for k, v in db.execute("SELECT key, value_json FROM trial_user_attributes WHERE trial_id=?", (trial_id,))}
                vector = db.execute("SELECT objective, value, value_type FROM trial_values WHERE trial_id=? ORDER BY objective", (trial_id,)).fetchall()
                values = [v for _, v, _ in vector]
                if [i for i, _, _ in vector] != list(range(len(metrics))) or any(t != "FINITE" for _, _, t in vector):
                    continue
                candidates.append(dict(number=number, values=values, params={k: decode_param(v, distribution) for k, v, distribution in db.execute("SELECT param_name,param_value,distribution_json FROM trial_params WHERE trial_id=?", (trial_id,))}, config=attrs.get("config"), slo_feasible=attrs.get("slo_feasible"), constraint_values=attrs.get("constraint_values"), measurement=attrs.get("measurement", {}), execution=attrs.get("execution", {}), state="COMPLETE", datetime_complete=end))
        coverage_label = "all local COMPLETE trials"
    elif record.get("remote_history"):
        candidates = [t for t in record["remote_history"] if t.get("state") == "COMPLETE"]
        coverage_label = "all W&B COMPLETE trials"
    else:
        candidates = record.get("remote_summary", {}).get("best_or_pareto_trials", [])
        coverage_label = "W&B candidate summary only"
    usable = 0
    for trial in candidates:
        values = trial.get("values", [])
        if len(values) != len(metrics) or any(seconds(v) is None for v in values):
            continue
        measurement = trial.get("measurement") or {}
        if measurement.get("benchmark_valid") is False:
            continue
        feasible = trial.get("slo_feasible")
        constraints = trial.get("constraint_values")
        if feasible is None and isinstance(constraints, list) and constraints:
            feasible = all(isinstance(v, (int, float)) and math.isfinite(v) and v <= 0 for v in constraints)
        if contract.get("constrained") and feasible is not True:
            continue
        env = contract.get("environment", {})
        context = dict(phase=record["phase"], framework=record["framework"], concurrency=contract.get("concurrency"),
            target=contract.get("target"), model=record["model_group"], workload=contract.get("workload"),
            gpu_type=env.get("gpu_type"), gpu_count=env.get("gpu_count"), package_versions=env.get("package_versions"),
            model_revision=env.get("model_revision"), cuda_version=env.get("cuda_version"), driver_version=env.get("driver_version"),
            constrained=contract.get("constrained"), metrics=metrics, directions=directions)
        if contract.get("constrained"):
            context.update(ttft_limit_ms=contract.get("ttft_limit_ms"), e2e_limit_ms=contract.get("e2e_limit_ms"))
        identity = record["wandb_url"] or record["database"] + "::" + record["study_name"]
        result_rows.append(dict(model_group=record["model_group"], run_group=record.get("run_group"), wandb_id=record["wandb_id"], wandb_url=record["wandb_url"], run_identity=identity, study_name=record["study_name"],
            phase=record["phase"], framework=record["framework"], concurrency=contract.get("concurrency"),
            target=contract.get("target"), sampler=contract.get("sampler"), model=contract.get("target_model"),
            context_key=json.dumps(context, sort_keys=True, default=str), metrics=metrics, directions=directions,
            trial_number=trial.get("number"), values=list(map(float, values)),
            config=trial.get("config"), params=trial.get("params"), slo_feasible=feasible,
            coverage=coverage_label, **dict(zip(metrics, map(float, values)))))
        usable += 1
    if not usable:
        result_issues.append({"study_name": record["study_name"], "issue": "No eligible completed objective results"})

results = pd.DataFrame(result_rows)
print(f"Loaded {len(results)} objective vectors from {results['run_identity'].nunique() if not results.empty else 0} included runs")

## Pareto frontier selection


In [ ]:
def frontier_indices(values, directions):
    """Return nondominated row positions; preserve tied configurations."""
    transformed = [[v if direction == "MINIMIZE" else -v for v, direction in zip(row, directions)] for row in values]
    return [i for i, row in enumerate(transformed) if not any(
        all(a <= b for a, b in zip(other, row)) and any(a < b for a, b in zip(other, row))
        for j, other in enumerate(transformed) if i != j)]


candidate_parts = []
if not results.empty:
    context_keys = sorted(results["context_key"].unique())
    context_ids = {key: f"C{i+1:02d}" for i, key in enumerate(context_keys)}
    results["context_id"] = results["context_key"].map(context_ids)
    for identity, frame in results.groupby("run_identity", sort=False):
        indices = frontier_indices(frame["values"].tolist(), frame.iloc[0]["directions"])
        candidate_parts.append(frame.iloc[indices])
    study_candidates = pd.concat(candidate_parts, ignore_index=True)
    candidate_counts = study_candidates.groupby("run_identity").size()
    study_result_summary = results.groupby("run_identity", sort=False).agg(
        study_name=("study_name", "first"), phase=("phase", "first"), framework=("framework", "first"),
        concurrency=("concurrency", "first"), objective=("target", "first"), sampler=("sampler", "first"),
        context_id=("context_id", "first"), coverage=("coverage", "first"), available_trials=("trial_number", "size"))
    study_result_summary["best_or_pareto_count"] = candidate_counts
    # Includes metric values and the actual winning/Pareto configurations for every run.
    result_contexts = pd.DataFrame([{"context_id": context_ids[k], **json.loads(k)} for k in context_keys])
else:
    study_candidates = pd.DataFrame()
    print("No eligible objective results available.")

#### Plot libraries and report labels


In [ ]:
import matplotlib.pyplot as plt
import re
import textwrap
import numpy as np
from IPython.display import Markdown

FRAMEWORK_NAMES = {"vllm": "vLLM", "trtllm": "TRT-LLM", "sglang": "SGLang"}
OBJECTIVE_ORDER = ["e2e", "ttft", "throughput", "pareto_latency", "pareto_throughput"]
OBJECTIVE_DESCRIPTION = {
    "e2e": "Minimize p95 end-to-end latency (ms)",
    "ttft": "Minimize p95 time to first token (ms)",
    "throughput": "Maximize output throughput (tokens/s)",
    "pareto_latency": "Minimize both p95 end-to-end latency and p95 TTFT (ms)",
    "pareto_throughput": "Maximize output throughput (tokens/s) and minimize p95 end-to-end latency (ms)",
}
OBJECTIVE_TITLES = {"e2e": "End-to-end latency", "ttft": "Time to first token",
                    "throughput": "Output throughput", "pareto_latency": "Latency Pareto frontier",
                    "pareto_throughput": "Throughput–latency Pareto frontier"}
METRIC_TITLES = {"p95_e2e_latency_ms": "p95 end-to-end latency (ms)",
                 "p95_ttft_ms": "p95 time to first token (ms)",
                 "output_throughput": "Output throughput (tokens/s)"}
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


#### Study repeat and seed labels


In [ ]:
def run_tokens(name):
    match = re.search(r"-s(\d+)-r(\d+)$", name)
    return (f"s{match.group(1)}", f"r{match.group(2)}") if match else ("seed unknown", "repeat unknown")


#### Included run note formatter


In [ ]:
def included_run_note(names, heading="Runs included"):
    names = sorted(set(names))
    seeds = sorted({run_tokens(name)[0] for name in names})
    repeats = sorted({run_tokens(name)[1] for name in names})
    lines = [f"**{heading}: {len(names)}.** Optimizer seeds: {', '.join(seeds)}. Repeats: {', '.join(repeats)}.",
             "`s` is the optimizer seed; `r` is the repeat number. Only the runs named below contribute."]
    lines.extend(f"- `{name}`" for name in names)
    return "\n\n".join(lines[:2]) + "\n\n" + "\n".join(lines[2:])


#### Figure display helper


In [ ]:
def finish_plot(fig):
    fig.tight_layout()
    plt.show()
    plt.close(fig)


### Run and objective selection

In [ ]:
def report_runs(phase, framework, concurrency=None, objective=None, model=None):
    return [r for r in included if r["phase"] == phase and r["framework"] == framework
            and (model is None or r.get("model_group") == model)
            and (concurrency is None or r["contract"].get("concurrency") == concurrency)
            and (objective is None or r["contract"].get("target") == objective)]


### Framework coverage plots

In [ ]:
def plot_framework_coverage(local_metadata, phase, framework):
    coverage = local_metadata.copy()
    coverage["target"] = coverage["target"].fillna("unknown")
    coverage["sampler_label"] = coverage["sampler"].fillna("unknown")
    targets = set(coverage["target"])
    objectives = [o for o in OBJECTIVE_ORDER if o in targets] + sorted(targets - set(OBJECTIVE_ORDER))
    concurrencies = sorted(coverage["concurrency"].dropna().unique())
    concurrency_labels = [f"c{int(c)}" for c in concurrencies]
    coverage["concurrency_label"] = coverage["concurrency"].map(lambda c: f"c{int(c)}" if pd.notna(c) else "unknown")
    if coverage["concurrency"].isna().any():
        concurrency_labels.append("unknown")
    counts = pd.crosstab(coverage["target"], coverage["concurrency_label"]).reindex(index=objectives, columns=concurrency_labels, fill_value=0)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    counts.plot.bar(ax=axes[0], rot=25)
    axes[0].set(title=f"Objective coverage · Phase {phase} · {FRAMEWORK_NAMES[framework]}", xlabel="Objective", ylabel="Included runs")
    axes[0].legend(title="Concurrency", fontsize=8)
    # Use one global sampler palette across all framework coverage plots.
    palette_names = sorted(set(metadata["sampler"].fillna("unknown")) | set(coverage["sampler_label"]))
    colors = {s: plt.get_cmap("tab10")(i % 10) for i, s in enumerate(palette_names)}
    positions = {objective: i for i, objective in enumerate(objectives)}
    for sampler, group in coverage.groupby("sampler_label", sort=True):
        group = group.sort_values(["target", "study_name"])
        x = group["target"].map(positions).to_numpy(dtype=float)
        # Spread repeated points slightly, without plotting any run twice.
        for objective in objectives:
            mask = group["target"].eq(objective).to_numpy()
            count = int(mask.sum())
            if count > 1:
                x[mask] += np.linspace(-0.13, 0.13, count)
        axes[1].scatter(x, group["duration_minutes"], color=colors[sampler], alpha=0.7, s=35, label=sampler)
    axes[1].axhline(MIN_RUN_SECONDS / 60, color="#dc2626", linestyle="--", label=f"Cutoff: {MIN_RUN_SECONDS / 60:g} min")
    axes[1].set(title=f"Run duration · Phase {phase} · {FRAMEWORK_NAMES[framework]}",
                xticks=range(len(objectives)), xticklabels=objectives, xlabel="Objective",
                ylabel="Optimization minutes (log scale)", yscale="log")
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].legend(title="Sampler / eligibility", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    return fig, axes

### Trial outcome summary

In [ ]:
def show_trial_outcomes(local_metadata):
    states = Counter()
    missing_runs = 0
    for row in local_metadata.to_dict("records"):
        recorded = row["trial_states"]
        if not isinstance(recorded, dict) or not recorded:
            missing_runs += 1
        else:
            states.update(recorded)
    display(pd.DataFrame([{"state": state, "trials": count}
                          for state, count in sorted(states.items())]))
    print(f"Runs without recorded outcome counts: {missing_runs}")

### Objective candidate table

In [ ]:
def objective_result_table(rows, frame, objective):
    metrics = rows[0]["contract"].get("metrics", []) if rows else []
    by_identity = {key: group for key, group in frame.groupby("run_identity", sort=False)} if not frame.empty else {}
    table_rows = []
    name_counts = Counter(run["study_name"] for run in rows)
    for run in rows:
        identity = run["wandb_url"] or run["database"] + "::" + run["study_name"]
        candidates = by_identity.get(identity)
        candidates = [None] if candidates is None else [candidate for _, candidate in candidates.iterrows()]
        for candidate in candidates:
            params = {} if candidate is None else candidate.get("params") or {}
            values = {} if candidate is None else {metric: candidate.get(metric) for metric in metrics}
            table_rows.append({
                "run": (f"{run['study_name']} [{run.get('run_group') or 'local'} / {run.get('wandb_id') or 'local'}]"
                        if name_counts[run["study_name"]] > 1 else run["study_name"]),
                "run_group": run.get("run_group"),
                "wandb_id": run.get("wandb_id"),
                "objective": run["contract"].get("target"),
                "context": None if candidate is None else candidate.get("context_id"),
                "sampler": run["contract"].get("sampler"),
                "runtime_minutes": None if run["duration_seconds"] is None else round(run["duration_seconds"] / 60, 2),
                "parameters": "<br>".join(
                    f"<b>{key}</b>: {html.escape(str(value))}"
                    for key, value in sorted(params.items())
                ) or "—",
                "trial": None if candidate is None else candidate.get("trial_number"),
                **{metric: values.get(metric) for metric in metrics},
                "SLO feasible": None if candidate is None else candidate.get("slo_feasible"),
                "result source": "no eligible result" if candidate is None else candidate.get("coverage"),
            })
    return pd.DataFrame(table_rows)

### Objective comparison plot

In [ ]:
def plot_objective(context_id, frame):
    first = frame.iloc[0]
    metrics, directions = first["metrics"], first["directions"]
    colors = {s: plt.get_cmap("tab10")(i % 10) for i, s in enumerate(sorted(frame["sampler"].fillna("unknown").unique()))}
    fig, ax = plt.subplots(figsize=(10, 5))
    if len(metrics) == 1:
        for position, (sampler, group) in enumerate(frame.groupby("sampler", dropna=False)):
            label = str(sampler) if pd.notna(sampler) else "unknown"
            group = group.drop_duplicates("run_identity")
            offsets = np.linspace(-0.15, 0.15, len(group)) if len(group) > 1 else np.zeros(1)
            ax.scatter(position + offsets, group[metrics[0]], color=colors[label], s=40)
            for x, (_, trial) in zip(position + offsets, group.iterrows()):
                ax.annotate(" / ".join(run_tokens(trial["study_name"])) + " / " + str(trial["wandb_id"] or "local"), (x, trial[metrics[0]]), xytext=(3, 4), textcoords="offset points", fontsize=8)
        labels = [str(s) if pd.notna(s) else "unknown" for s, _ in frame.groupby("sampler", dropna=False)]
        ax.set(xticks=range(len(labels)), xticklabels=labels, xlabel="Sampler; dots are individual runs",
               ylabel=f"{METRIC_TITLES.get(metrics[0], metrics[0])} — {directions[0].lower()}")
    elif len(metrics) == 2:
        for identity, group in frame.groupby("run_identity", sort=False):
            ordered = group.sort_values(metrics[0])
            label = f"{ordered.iloc[0]['sampler']} / {' / '.join(run_tokens(ordered.iloc[0]['study_name']))} / {ordered.iloc[0]['wandb_id'] or 'local'}"
            sampler = ordered.iloc[0]["sampler"] or "unknown"
            ax.plot(ordered[metrics[0]], ordered[metrics[1]], "o-", color=colors[sampler], alpha=0.7, label=label)
        ax.set(xlabel=f"{METRIC_TITLES.get(metrics[0], metrics[0])} — {directions[0].lower()}", ylabel=f"{METRIC_TITLES.get(metrics[1], metrics[1])} — {directions[1].lower()}")
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.set_title(f"{OBJECTIVE_TITLES.get(first['target'], first['target'])}\nPhase {first['phase']} · {FRAMEWORK_NAMES[first['framework']]} · concurrency {first['concurrency']} · {context_id}")
    # Embed exact study names in the figure, so exported plots retain provenance.
    caption = "Included runs: " + "; ".join(sorted({f"{r.study_name} [{r.run_group}; {r.wandb_id or 'local'}]" for r in frame.itertuples()}))
    lines = textwrap.wrap(caption, width=105)
    fig.text(0.02, 0.01, "\n".join(lines), ha="left", va="bottom", fontsize=7)
    fig.tight_layout(rect=(0, 0.06 + 0.018 * len(lines), 1, 1))
    plt.show()
    plt.close(fig)
    return

### Objective values over trials

In [ ]:
def plot_objective_trials(rows, objective, context_id):
    identities = {r["wandb_url"] or r["database"] + "::" + r["study_name"] for r in rows}
    history = results[(results["run_identity"].isin(identities)) & (results["context_id"] == context_id)] if not results.empty else pd.DataFrame()
    if history.empty:
        print("Trial history is unavailable for these W&B summary-only results.")
        return
    metrics = history.iloc[0]["metrics"]
    directions = history.iloc[0]["directions"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(7 * len(metrics), 4), squeeze=False)
    samplers = sorted(history["sampler"].fillna("unknown").unique())
    colors = {sampler: plt.get_cmap("tab10")(i % 10) for i, sampler in enumerate(samplers)}
    for ax, metric, direction in zip(axes.flat, metrics, directions):
        for identity, run_history in history.groupby("run_identity", sort=False):
            run_history = run_history.sort_values("trial_number")
            first = run_history.iloc[0]
            label = f"{first['sampler']} · {run_tokens(first['study_name'])[1]} · {first['wandb_id'] or 'local'}"
            values = run_history[metric]
            ax.scatter(run_history["trial_number"], values, color=colors[first["sampler"] or "unknown"], alpha=0.5, s=24)
            best = values.cummin() if direction == "MINIMIZE" else values.cummax()
            ax.step(run_history["trial_number"], best, where="post", color=colors[first["sampler"] or "unknown"], label=label)
        ax.set(title=METRIC_TITLES.get(metric, metric), xlabel="Trial number",
               ylabel=f"{METRIC_TITLES.get(metric, metric)} — {direction.lower()}")
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.suptitle(f"Objective values over trials · {objective} · {context_id}")
    finish_plot(fig)

### Retrospective Optuna feature importance

In [ ]:
def parameter_importance(rows, objective):
    try:
        import optuna
    except ImportError:
        print("Feature importance unavailable: install Optuna.")
        return pd.DataFrame()
    importance_rows = []
    for run in rows:
        if not run.get("database"):
            try:
                scores, reason = history_importance(run, run.get("remote_history", []))
                importance_rows.extend(scores)
                if reason:
                    importance_coverage.append({"model_group": run["model_group"], "framework": run["framework"],
                        "run": run["study_name"], "wandb_id": run["wandb_id"], "status": reason})
            except (ImportError, ValueError, KeyError, RuntimeError) as exc:
                importance_coverage.append({"model_group": run["model_group"], "framework": run["framework"],
                    "run": run["study_name"], "wandb_id": run["wandb_id"], "status": str(exc)})
            continue
        contract = run["contract"]
        metrics = contract.get("metrics", [])
        try:
            study = optuna.load_study(
                study_name=run["study_name"],
                storage=f"sqlite:///{(ROOT / run['database']).resolve()}",
            )
            for index, metric in enumerate(metrics):
                values = [trial.values[index] for trial in study.trials
                          if trial.state.name == "COMPLETE" and trial.values
                          and index < len(trial.values) and math.isfinite(trial.values[index])]
                if len(values) < 2 or len(set(values)) < 2:
                    importance_coverage.append({"model_group": run["model_group"], "framework": run["framework"],
                        "run": run["study_name"], "wandb_id": run["wandb_id"], "status": f"{metric}: insufficient variation"})
                    continue
                direction = contract["directions"][index].upper()
                sign = 1 if direction == "MINIMIZE" else -1
                evaluator = optuna.importance.PedAnovaImportanceEvaluator()
                scores = optuna.importance.get_param_importances(
                    study,
                    target=lambda trial, i=index, multiplier=sign: multiplier * trial.values[i],
                    evaluator=evaluator,
                )
                for rank, (param, score) in enumerate(scores.items(), 1):
                    importance_rows.append({"model_group": run["model_group"], "framework": run["framework"], "target": contract.get("target"), "wandb_id": run["wandb_id"], "source": "local SQLite", "evaluator": "Optuna PED-ANOVA", "run": run["study_name"], "sampler": contract.get("sampler"),
                        "metric": metric, "rank": rank, "parameter": param, "importance": score})
        except (ImportError, ValueError, KeyError, RuntimeError) as exc:
            result_issues.append({"study_name": run["study_name"],
                                  "issue": f"Retrospective importance unavailable ({type(exc).__name__})"})
    return pd.DataFrame(importance_rows)


def show_parameter_importance(rows, objective):
    importance = parameter_importance(rows, objective)
    if importance.empty:
        print("No local runs have enough varied completed trials for retrospective importance.")
        print("W&B summary-only runs do not contain trial histories for this calculation.")
        return
    display(importance.sort_values(["run", "metric", "rank"]))
    for metric, frame in importance.groupby("metric", sort=False):
        fig, ax = plt.subplots(figsize=(9, max(3, 0.3 * len(frame))))
        for run, group in frame.groupby("run", sort=False):
            ax.plot(group["importance"], group["parameter"], "o", label=f"{run} / {group.iloc[0]['sampler']}")
        ax.set(title=f"Retrospective Optuna importance · {metric}", xlabel="Importance", ylabel="Trial parameter")
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
        finish_plot(fig)

### Matching-context results

In [ ]:
def show_context_results(rows, objective, context_id):
    ids = {r["wandb_url"] or r["database"] + "::" + r["study_name"] for r in rows}
    frame = study_candidates[study_candidates["run_identity"].isin(ids)] if not study_candidates.empty else pd.DataFrame()
    frame = frame[frame["context_id"] == context_id] if not frame.empty else frame
    display(Markdown(f"##### Matching context {context_id}"))
    if frame.empty:
        display(objective_result_table(rows, frame, objective))
        print("No eligible objective candidates for this context.")
        return
    context = result_contexts[result_contexts["context_id"] == context_id].iloc[0]
    fields = ["model", "concurrency", "workload", "gpu_type", "gpu_count",
              "model_revision", "cuda_version", "driver_version"]
    details = [f"**{field.replace('_', ' ')}:** `{json.dumps(context[field], sort_keys=True, default=lambda value: value.item() if hasattr(value, 'item') else str(value))}`"
               for field in fields if field in context and context[field] is not None]
    display(Markdown("**Matched workload and environment**\n\n" + " · ".join(details)))
    plot_objective(context_id, frame)
    plot_objective_trials(rows, objective, context_id)

### Objective section

In [ ]:
def show_objective(phase, framework, concurrency, objective, model=None):
    rows = report_runs(phase, framework, concurrency, objective, model)
    display(Markdown(f"##### {OBJECTIVE_TITLES.get(objective, objective)} (`{objective}`)"))
    display(Markdown(OBJECTIVE_DESCRIPTION.get(objective, "Use the stored objective contract.")))
    display(Markdown(included_run_note([run["study_name"] for run in rows])))
    identities = {r["wandb_url"] or r["database"] + "::" + r["study_name"] for r in rows}
    candidates = study_candidates[study_candidates["run_identity"].isin(identities)] if not study_candidates.empty else pd.DataFrame()
    if candidates.empty:
        print("No eligible objective results; see audit issues.")
    else:
        display(HTML(objective_result_table(rows, candidates, objective).to_html(escape=False, index=False)))
        for context_id in sorted(candidates["context_id"].unique()):
            show_context_results(rows, objective, context_id)


### Concurrency section

In [ ]:
def show_concurrency(phase, framework, rows, concurrency, model=None):
    display(Markdown(f"#### Concurrency {concurrency} · Phase {phase} · {FRAMEWORK_NAMES[framework]}"))
    objectives = {run["contract"].get("target") for run in rows
                  if run["contract"].get("concurrency") == concurrency}
    ordered = [obj for obj in OBJECTIVE_ORDER if obj in objectives]
    ordered += sorted(obj for obj in objectives if obj not in OBJECTIVE_ORDER and obj is not None)
    for objective in ordered:
        show_objective(phase, framework, concurrency, objective, model)

### Framework section

In [ ]:
def show_framework(phase, framework, model=None):
    rows = report_runs(phase, framework, model=model)
    print(f"Phase {phase} / {FRAMEWORK_NAMES[framework]}: {len(rows)} included runs; cutoff {MIN_RUN_SECONDS / 60:g} minutes")
    display(Markdown(included_run_note([run["study_name"] for run in rows])))
    if not rows:
        return
    local_metadata = pd.DataFrame([metadata_row(run) for run in rows])
    fig, _ = plot_framework_coverage(local_metadata, phase, framework)
    finish_plot(fig)
    show_trial_outcomes(local_metadata)
    concurrencies = sorted({run["contract"].get("concurrency") for run in rows},
                           key=lambda value: (value is None, value or 0))
    for concurrency in concurrencies:
        show_concurrency(phase, framework, rows, concurrency, model)

### Early Phase A vLLM latency inventory

In [ ]:
def early_vllm_latency_runs():
    return [r for r in report_runs("A", "vllm")
            if run_tokens(r["study_name"])[1] == "r1"
            and r["contract"].get("target") in {"e2e", "ttft", "pareto_latency"}]

def show_early_vllm_latency():
    rows = early_vllm_latency_runs()
    display(Markdown("#### Phase A vLLM: early and first-repeat latency runs\n\nOriginal groups and W&B IDs distinguish runs sharing an `r1` study name. The earliest concurrency-128 E2E group is `optuna-inference-lab-idcgg3zn`; `optuna-inference-lab-p0lfaev2` contains later same-name repeats. These rows use the same five-minute inclusion rule and remain in the objective sections below; they are not counted twice."))
    if study_candidates.empty:
        return
    identities = {r["wandb_url"] or r["database"] + "::" + r["study_name"] for r in rows}
    selected = study_candidates[study_candidates["run_identity"].isin(identities)]
    display(Markdown("**Recorded best / Pareto results for these r1 runs**\n\nRemote-only entries are W&B candidate summaries; local entries use completed SQLite trials. Compare within the listed matching context, not across workloads."))
    if not selected.empty:
        columns = ["study_name", "run_group", "wandb_id", "wandb_url", "concurrency", "target", "context_id", "trial_number", "p95_e2e_latency_ms", "p95_ttft_ms", "coverage", "config"]
        result_tables = []
        for objective in sorted(selected["target"].unique()):
            objective_rows = [r for r in rows if r["contract"].get("target") == objective]
            identities = {r["wandb_url"] or r["database"] + "::" + r["study_name"] for r in objective_rows}
            objective_candidates = selected[selected["run_identity"].isin(identities)]
            result_tables.append(objective_result_table(objective_rows, objective_candidates, objective))
        display(pd.concat(result_tables, ignore_index=True).sort_values(["objective", "run_group", "run"]))

## Model reports
Each model section includes objective mean/variance, initiation and loading mean/variance, readable trial parameter tables, per-framework Optuna importance, and the existing context-specific objective plots. Missing model/framework data is reported explicitly.


In [ ]:
# Cache timing and importance once, avoiding repeated analysis per concurrency.
importance_coverage = []
all_importance = parameter_importance(included, None)
all_objective_statistics = objective_statistics(results)
startup_data, startup_coverage = [], []
for run in records:
    if exclusion(run) in {"synthetic", "missing or unsupported framework"}:
        continue
    trials = run.get("remote_history", [])
    timing_source = "W&B trial history"
    if run.get("database"):
        timing_source = "local SQLite"
        trials = []
        with closing(sqlite3.connect((ROOT / run["database"]).resolve().as_uri() + "?mode=ro", uri=True)) as db:
            study_id = db.execute("SELECT study_id FROM studies WHERE study_name=?", (run["study_name"],)).fetchone()[0]
            for trial_id, number, state in db.execute("SELECT trial_id, number, state FROM trials WHERE study_id=?", (study_id,)):
                attrs = {k: json.loads(v) for k, v in db.execute("SELECT key, value_json FROM trial_user_attributes WHERE trial_id=?", (trial_id,))}
                params = {k: decode_param(v, d) for k, v, d in db.execute("SELECT param_name,param_value,distribution_json FROM trial_params WHERE trial_id=?", (trial_id,))}
                trials.append(dict(number=number, state=state, params=params, execution=attrs.get("execution", {}), measurement=attrs.get("measurement", {})))
    timing = startup_rows(run, trials, timing_source)
    # Remote histories can supplement a local database without execution metadata.
    if not timing and run.get("database"):
        timing = startup_rows(run, run.get("remote_history", []), "W&B trial history")
    startup_data.extend(timing)
    startup_coverage.append(dict(model_group=run["model_group"], framework=run["framework"],
        run=run.get("display_name", run["study_name"]), wandb_id=run["wandb_id"],
        timing_observations=len(timing), objective_inclusion=exclusion(run) or "included"))
startup_details = pd.DataFrame(startup_data)
all_startup_statistics = startup_statistics(startup_details)


def show_model(model):
    rows = [r for r in included if r["model_group"] == model]
    display(Markdown(f"**{len(rows)} eligible optimization runs**"))
    display(manifest(rows))
    display(Markdown("### Objective means and sample variances"))
    if not all_objective_statistics.empty:
        display(all_objective_statistics[all_objective_statistics["model_group"] == model])
    display(Markdown("### Server initiation and model loading"))
    coverage = pd.DataFrame(startup_coverage)
    display(coverage[coverage["model_group"] == model])
    if not all_startup_statistics.empty:
        display(all_startup_statistics[all_startup_statistics["model_group"] == model])
    detail = startup_details[startup_details["model_group"] == model] if not startup_details.empty else pd.DataFrame()
    if detail.empty:
        print("No recorded startup telemetry available for this model.")
    else:
        display(Markdown("Each row is one trial; parameters wrap onto separate lines and repeated engine-log durations appear as lists. Launch-to-ready includes model loading. Log observations may overlap. Partial phases are excluded from means."))
        display(HTML(table_html(startup_trial_table(detail))))
    for framework in FRAMEWORK_NAMES:
        selected = [r for r in rows if r["framework"] == framework]
        display(Markdown(f"### {FRAMEWORK_NAMES[framework]} · {model}"))
        if not selected:
            print("No eligible runs available.")
            continue
        display(Markdown("#### Optuna feature importance"))
        scores = all_importance[(all_importance["model_group"] == model) & (all_importance["framework"] == framework)] if not all_importance.empty else pd.DataFrame()
        if scores.empty:
            print("Optuna importance unavailable; see coverage reasons below.")
        else:
            display(scores.sort_values(["run", "wandb_id", "metric", "rank"]))
            # Average per-run importances only within the same objective metric.
            # This descriptive summary does not refit a pooled search space.
            display(scores.groupby(["target", "metric", "parameter"], dropna=False).agg(
                runs=("importance", "size"), mean_importance=("importance", "mean"),
                variance=("importance", "var")).reset_index())
        missing = pd.DataFrame(importance_coverage)
        if not missing.empty:
            display(missing[(missing["model_group"] == model) & (missing["framework"] == framework)])
        for phase in sorted({r["phase"] for r in selected}, reverse=True):
            if any(r["phase"] == phase for r in selected):
                show_framework(phase, framework, model)


## cpm5_2


In [ ]:
show_model("cpm5_2")


## qwen3.5-4b


In [ ]:
show_model("qwen3.5-4b")


## qwen3-4b


In [ ]:
show_model("qwen3-4b")


## qwen3-4b dspark


In [ ]:
show_model("qwen3-4b dspark")


## qwen3-8b


In [ ]:
show_model("qwen3-8b")


## qwen3-8b dspark


In [ ]:
show_model("qwen3-8b dspark")


## gemma4-12b


In [ ]:
show_model("gemma4-12b")


## gemma4-12b dspark


In [ ]:
show_model("gemma4-12b dspark")


## Additional variants


In [ ]:
for model in sorted({r["model_group"] for r in records} - set(MODEL_ORDER)):
    display(Markdown(f"## {model}"))
    show_model(model)


## Audit: exclusions and source coverage

These tables explain runs that did not contribute. No excluded run is used by the framework plots. W&B-only candidate summaries cannot establish full search histories.

### Excluded runs and source issues


In [ ]:
print(f"Excluded runs: {len(excluded)}")
excluded_runs = manifest(excluded)
excluded_runs["exclusion_reason"] = [exclusion(r) for r in excluded]
display(excluded_runs)
source_issues = pd.DataFrame(issues, columns=["source", "issue"])
display(source_issues)

display(pd.DataFrame(result_issues, columns=["study_name", "issue"]))

## Save a readable HTML report
This standalone report contains the model summaries, importance tables and per-trial initiation parameters. It can be opened directly in a browser.


In [ ]:
HTML_REPORT = ROOT / "docs" / "optuna_run_metadata.html"
write_report_html(
    HTML_REPORT,
    MODEL_ORDER + sorted({r["model_group"] for r in records} - set(MODEL_ORDER)),
    runs_used, all_objective_statistics, all_startup_statistics, startup_details,
    pd.DataFrame(startup_coverage), all_importance, pd.DataFrame(importance_coverage),
    excluded_runs, source_issues, exported_at=exported_at if WANDB_SNAPSHOT.exists() else None,
)
print(f"Saved {HTML_REPORT.relative_to(ROOT)}")
